<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=344464862" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== STAGE 16 SESSION: GENERATE FOLD ASSIGNMENTS (once, ever) + FOLD 1, CUSTOM CNN + EFFICIENTNETB0 =====
# Dual checkpoints per architecture: val_auc controls EarlyStopping and is the primary reported
# result; val_accuracy is tracked as a secondary checkpoint for comparison, costs no extra
# training time, just a second saved file and a second quick evaluation.

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (cohen_kappa_score, roc_auc_score, accuracy_score,
                              f1_score, recall_score, confusion_matrix)

# ---------- CONFIG ----------
APTOS_CSV     = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG     = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images'
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'

GRADES, NUM_CLASSES = ['0','1','2','3','4'], 5
IMG_SIZE, BATCH_SIZE = 224, 32
SUBSAMPLE_SEED, EYEPACS_TARGET = 42, 3662
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, CUSTOM_LR, EARLYSTOP_PAT = 10, 1e-3, 1e-5, 1e-3, 7
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
CURRENT_FOLD = 1

# ================================================================
# PART A: BUILD POOLED DATAFRAME WITH group_id (three different grouping situations)
# ================================================================
aptos = pd.read_csv(APTOS_CSV)
aptos['grade']      = aptos['diagnosis'].astype(int).astype(str)
aptos['image_path'] = APTOS_IMG + '/' + aptos['id_code'].astype(str) + '.png'
aptos['source']     = 'aptos'
aptos['group_id']   = 'aptos_row' + aptos.index.astype(str)  # no grouping signal, singleton per row

eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['source']     = 'eyepacs'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')

def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)
eyepacs_s['group_id'] = 'eyepacs_pat' + eyepacs_s['patient_id'].astype(str)  # real patient groups

messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['source']     = 'messidor'

is_im = ~messidor['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
im_df = messidor[is_im].copy()
im_df['im_num'] = im_df['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
im_df = im_df.sort_values('im_num').reset_index(drop=True)
im_df['group_id'] = 'messidor_pair' + (im_df.index // 2).astype(str)  # pair-derived groups, 74.9% reliable
date_df = messidor[~is_im].copy().reset_index(drop=True)
date_df['group_id'] = 'messidor_row' + date_df.index.astype(str)  # no signal, singleton per row
messidor_grouped = pd.concat([im_df.drop(columns=['im_num']), date_df], ignore_index=True)

cols = ['image_path', 'grade', 'source', 'group_id']
pooled = pd.concat([aptos[cols], eyepacs_s[cols], messidor_grouped[cols]], ignore_index=True)
print(f"Pooled: {len(pooled):,} rows (expect 9,068)")
assert len(pooled) == 9068

ungrouped = pooled['group_id'].str.contains('_row').sum()
print(f"Ungrouped rows (singleton group_id, no real grouping signal): {ungrouped:,} "
      f"({ungrouped/len(pooled)*100:.1f}% of pooled, expect ~52%)")

# ================================================================
# PART B: GENERATE 5-FOLD ASSIGNMENT, ONCE. Stratified per source, group-safe.
# ================================================================
pooled['fold'] = -1
for src in ['aptos', 'eyepacs', 'messidor']:
    sub = pooled[pooled['source'] == src].reset_index()
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold_num, (_, test_idx) in enumerate(sgkf.split(sub, sub['grade'], sub['group_id']), start=1):
        orig_idx = sub.loc[test_idx, 'index']
        pooled.loc[orig_idx, 'fold'] = fold_num
assert (pooled['fold'] != -1).all(), "Some rows never got a fold assignment"

print("\n===== FOLD ASSIGNMENT VERIFICATION =====")
print("Fold sizes:")
print(pooled['fold'].value_counts().sort_index())

print("\nGrade coverage per fold (rare grades 3, 4 checked explicitly):")
for f in range(1, 6):
    counts = pooled[pooled.fold == f]['grade'].value_counts()
    print(f"  Fold {f}: n={len(pooled[pooled.fold==f])}, grade3={counts.get('3',0)}, "
          f"grade4={counts.get('4',0)}, full={dict(counts.sort_index())}")

print("\nSource coverage per fold:")
print(pd.crosstab(pooled['fold'], pooled['source']))

leak = pooled.groupby('group_id')['fold'].nunique()
assert (leak == 1).all(), f"LEAKAGE: {(leak>1).sum()} group_ids span multiple folds"
print(f"\nGroup-level leakage check: PASS, all {len(leak):,} groups map to exactly one fold")

pooled.to_csv('/kaggle/working/dr_cv_fold_assignments.csv', index=False)
print("\nSaved dr_cv_fold_assignments.csv")
print("="*70)
print("DOWNLOAD THIS FILE NOW AND UPLOAD IT AS A KAGGLE DATASET.")
print("Every future Stage 16 session (fold 1 mob/res, folds 2 through 5) must LOAD this exact")
print("file. Never regenerate it. A regenerated partition risks a subtly different split if any")
print("upstream code changes, which would silently break cross-fold comparability with no error.")
print("="*70)

# ================================================================
# PART C: FOLD 1 SETUP, inner group-level train/val split from the other 4 folds
# ================================================================
test_df   = pooled[pooled.fold == CURRENT_FOLD].reset_index(drop=True)
remainder = pooled[pooled.fold != CURRENT_FOLD].reset_index(drop=True)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, unstratified fallback. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_group_level_inner(df, label_col='grade', val_frac=0.15, rs=SEED, tag=""):
    grouped = df.groupby('group_id')[label_col].agg(lambda s: s.value_counts().index[0]).reset_index()
    g_tr, g_va = safe_split(grouped, label_col, val_frac, rs, tag=tag)
    pick = lambda ids: df[df['group_id'].isin(ids['group_id'])]
    return pick(g_tr), pick(g_va)

train_parts, val_parts = [], []
for src in ['aptos', 'eyepacs', 'messidor']:
    sub = remainder[remainder.source == src]
    tr_s, va_s = split_group_level_inner(sub, tag=f"fold{CURRENT_FOLD}-{src}-inner")
    train_parts.append(tr_s); val_parts.append(va_s)
train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts, ignore_index=True)

print(f"\nFold {CURRENT_FOLD}: Train {len(train_df):,} ({len(train_df)/len(pooled)*100:.1f}%) | "
      f"Val {len(val_df):,} ({len(val_df)/len(pooled)*100:.1f}%) | "
      f"Test {len(test_df):,} ({len(test_df)/len(pooled)*100:.1f}%)")
print("(Proportions drift slightly from 68/12/20, the inner split selects groups, not rows.)")

for src in ['aptos', 'eyepacs', 'messidor']:
    tr_g = set(train_df[train_df.source==src]['group_id'])
    va_g = set(val_df[val_df.source==src]['group_id'])
    te_g = set(test_df[test_df.source==src]['group_id'])
    ok = tr_g.isdisjoint(va_g) and tr_g.isdisjoint(te_g) and va_g.isdisjoint(te_g)
    print(f"  {src}: train/val/test group-disjoint = {ok}")
    assert ok, f"LEAKAGE in fold {CURRENT_FOLD}, source {src}"
print(f"Fold {CURRENT_FOLD} leakage check: PASS (aptos, eyepacs, messidor)")

cls = np.array(GRADES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['grade'])
CLASS_WEIGHT = {i: w for i, w in enumerate(cw)}
print(f"Fold {CURRENT_FOLD} class_weight (fresh from this fold's train set):",
      {c: round(w,3) for c,w in zip(cls,cw)}, f"| span {cw.max()/cw.min():.1f}x")

def make_gens(preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                  class_mode='categorical', classes=GRADES, color_mode='rgb')
    return (train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, **common),
            eval_idg.flow_from_dataframe(val_df, shuffle=False, **common),
            eval_idg.flow_from_dataframe(test_df, shuffle=False, **common))

def build_custom_cnn(num_classes=5, shape=(224,224,3)):
    return Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])

def build_pretrained(base_class, num_classes=5, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                        Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def macro_specificity(y_true, y_pred, n_classes=NUM_CLASSES):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp) > 0 else np.nan)
    return np.nanmean(specs)

def full_test_metrics(model, te_gen):
    y_prob = model.predict(te_gen, verbose=0)
    y_true = np.asarray(te_gen.classes)
    y_pred = y_prob.argmax(axis=1)
    try:
        auc = roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr')
    except ValueError:
        auc = np.nan
    return dict(qwk=cohen_kappa_score(y_true, y_pred, weights='quadratic'), macro_auc=auc,
                accuracy=accuracy_score(y_true, y_pred),
                macro_f1=f1_score(y_true, y_pred, average='macro'),
                macro_sensitivity=recall_score(y_true, y_pred, average='macro'),
                macro_specificity=macro_specificity(y_true, y_pred), n_test=len(y_true)), y_true, y_pred, y_prob

# ================================================================
# PART D: TRAIN ONE ARCHITECTURE, DUAL CHECKPOINTS, STANDALONE CSV OUTPUT
# ================================================================
def train_and_evaluate(arch_code, build_fn, preprocess_fn, is_pretrained):
    tr, va, te = make_gens(preprocess_fn)
    auc_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_aucbest.keras'
    acc_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_accbest.keras'

    if is_pretrained:
        model, base = build_fn()
        base.trainable = False
        model.compile(Adam(PHASE1_LR), 'categorical_crossentropy',
                      metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
        print(f"\n===== Fold {CURRENT_FOLD}, {arch_code}: PHASE 1 (head only, {PHASE1_EPOCHS} epochs) =====")
        model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=CLASS_WEIGHT, verbose=1,
                  callbacks=[CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_log.csv', append=False)])
        base.trainable = True
        model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
                      metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
        print(f"\n===== Fold {CURRENT_FOLD}, {arch_code}: PHASE 2 (full fine-tune, dual checkpoint) =====")
        hist = model.fit(tr, validation_data=va, epochs=60, class_weight=CLASS_WEIGHT, verbose=1,
                  callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=EARLYSTOP_PAT, restore_best_weights=True),
                             ModelCheckpoint(auc_path, monitor='val_auc', mode='max', save_best_only=True),
                             ModelCheckpoint(acc_path, monitor='val_accuracy', mode='max', save_best_only=True),
                             CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_log.csv', append=True)])
    else:
        model = build_fn()
        model.compile(Adam(CUSTOM_LR), 'categorical_crossentropy',
                      metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
        print(f"\n===== Fold {CURRENT_FOLD}, {arch_code}: single phase, dual checkpoint =====")
        hist = model.fit(tr, validation_data=va, epochs=60, class_weight=CLASS_WEIGHT, verbose=1,
                  callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=EARLYSTOP_PAT, restore_best_weights=True),
                             ModelCheckpoint(auc_path, monitor='val_auc', mode='max', save_best_only=True),
                             ModelCheckpoint(acc_path, monitor='val_accuracy', mode='max', save_best_only=True),
                             CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_log.csv', append=False)])

    # ---- PRIMARY: val_auc-selected checkpoint. EarlyStopping's restore_best_weights matches
    # this monitor, so the in-memory model right now genuinely IS the auc-best checkpoint,
    # a live in-memory metric is a valid provenance check against the reloaded file. ----
    live_metrics, y_true, y_pred, y_prob = full_test_metrics(model, te)
    print(f"\nLive (in-memory) test metrics, auc-selected: {live_metrics}")
    np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_preds.npz',
             y_true=y_true, y_pred=y_pred, y_prob=y_prob,
             source=test_df['source'].values, group_id=test_df['group_id'].values)

    reloaded_auc_model = load_model(auc_path)
    reloaded_auc_metrics, _, _, _ = full_test_metrics(reloaded_auc_model, te)
    match = abs(live_metrics['macro_auc'] - reloaded_auc_metrics['macro_auc']) < 1e-3
    print(f"Reloaded auc-checkpoint macro_auc={reloaded_auc_metrics['macro_auc']:.4f} vs live={live_metrics['macro_auc']:.4f}, match={match}")
    assert match, "AUC-CHECKPOINT MISMATCH, invalid provenance, discard this result"
    del reloaded_auc_model

    # ---- SECONDARY: val_accuracy-selected checkpoint. EarlyStopping tracked val_auc, not
    # val_accuracy, so the in-memory model does NOT correspond to this checkpoint. Provenance
    # is instead checked against the VALIDATION-set history the accuracy monitor actually saw
    # during training, the only independent reference available for this checkpoint. ----
    best_val_acc_seen = max(hist.history['val_accuracy'])
    reloaded_acc_model = load_model(acc_path)
    reloaded_val_acc = reloaded_acc_model.evaluate(va, verbose=0)[1]
    match_acc = abs(best_val_acc_seen - reloaded_val_acc) < 1e-3
    print(f"Accuracy-checkpoint provenance: best val_accuracy seen during training={best_val_acc_seen:.4f} "
          f"vs reloaded val_accuracy={reloaded_val_acc:.4f}, match={match_acc}")
    assert match_acc, "ACCURACY-CHECKPOINT MISMATCH, invalid provenance, discard this result"
    reloaded_acc_metrics, _, _, _ = full_test_metrics(reloaded_acc_model, te)
    del reloaded_acc_model
    del model; import gc; gc.collect(); tf.keras.backend.clear_session()

    rows = [
        {'fold': CURRENT_FOLD, 'arch': arch_code, 'selected_by': 'val_auc', **reloaded_auc_metrics},
        {'fold': CURRENT_FOLD, 'arch': arch_code, 'selected_by': 'val_accuracy', **reloaded_acc_metrics},
    ]
    out_df = pd.DataFrame(rows)
    out_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_results.csv'
    out_df.to_csv(out_path, index=False)
    print(f"\nSaved {out_path}")
    print(out_df.round(4).to_string(index=False))
    return out_df

# ---- CUSTOM CNN ----
custom_results = train_and_evaluate('custom', build_custom_cnn, None, is_pretrained=False)

# ---- EFFICIENTNETB0 ----
eff_results = train_and_evaluate('eff', lambda: build_pretrained(EfficientNetB0), eff_pre, is_pretrained=True)

print(f"\n\n{'='*20} FOLD {CURRENT_FOLD}: CUSTOM + EFF DONE (2 of 4) {'='*20}")
print(pd.concat([custom_results, eff_results], ignore_index=True).round(4).to_string(index=False))
print(f"\nDownload individually: cv_f{CURRENT_FOLD}_custom_results.csv, cv_f{CURRENT_FOLD}_eff_results.csv")
print(f"Plus: dr_cv_fold_assignments.csv (upload as a new Kaggle dataset before the next session)")
print(f"Still needed for fold {CURRENT_FOLD}: mob, res. Do not start fold 2 until all 4 exist.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 63.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-23 22:18:22.864992: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787523502.887818      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787523502.895354      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787523502.914121      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787523502.914140      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787523502.914143      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Pooled: 9,068 rows (expect 9,068)
Ungrouped rows (singleton group_id, no real grouping signal): 4,719 (52.0% of pooled, expect ~52%)

===== FOLD ASSIGNMENT VERIFICATION =====
Fold sizes:
fold
1    1811
2    1812
3    1816
4    1815
5    1814
Name: count, dtype: int64

Grade coverage per fold (rare grades 3, 4 checked explicitly):
  Fold 1: n=1811, grade3=83, grade4=80, full={'0': np.int64(1110), '1': np.int64(161), '2': np.int64(377), '3': np.int64(83), '4': np.int64(80)}
  Fold 2: n=1812, grade3=73, grade4=93, full={'0': np.int64(1064), '1': np.int64(183), '2': np.int64(399), '3': np.int64(73), '4': np.int64(93)}
  Fold 3: n=1816, grade3=61, grade4=69, full={'0': np.int64(1133), '1': np.int64(179), '2': np.int64(374), '3': np.int64(61), '4': np.int64(69)}
  Fold 4: n=1815, grade3=64, grade4=84, full={'0': np.int64(1118), '1': np.int64(182), '2': np.int64(367), '3': np.int64(64), '4': np.int64(84)}
  Fold 5: n=1814, grade3

I0000 00:00:1787523534.868490      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787523534.874557      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



===== Fold 1, custom: single phase, dual checkpoint =====
Epoch 1/60


...

193/193 [==============================] - 530s 3s/step - loss: 1.6616 - accuracy: 0.2933 - auc: 0.5937 - val_loss: 2.3321 - val_accuracy: 0.1493 - val_auc: 0.3101
Epoch 2/60
193/193 [==============================] - 417s 2s/step - loss: 1.5283 - accuracy: 0.4287 - auc: 0.7165 - val_loss: 2.2816 - val_accuracy: 0.0366 - val_auc: 0.2426
Epoch 3/60
193/193 [==============================] - 420s 2s/step - loss: 1.4991 - accuracy: 0.4788 - auc: 0.7529 - val_loss: 1.7661 - val_accuracy: 0.1978 - val_auc: 0.4290
Epoch 4/60
193/193 [==============================] - 419s 2s/step - loss: 1.4871 - accuracy: 0.5066 - auc: 0.7615 - val_loss: 1.6064 - val_accuracy: 0.2830 - val_auc: 0.5822
Epoch 5/60
193/193 [==============================] - 418s 2s/step - loss: 1.4695 - accuracy: 0.5100 - auc: 0.7745 - val_loss: 1.2177 - val_accuracy: 0.6126 - val_auc: 0.8415
Epoch 6/60
193/193 [==============================] - 420s 2s/step - loss: 1.4586 - accuracy: 0.5301 - auc: 0.7890 - val_loss: 1.3924 - 

E0000 00:00:1787529844.544377      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


193/193 [==============================] - 393s 2s/step - loss: 1.3878 - accuracy: 0.4808 - auc: 0.7895 - val_loss: 1.1726 - val_accuracy: 0.4643 - val_auc: 0.8175
Epoch 2/10
193/193 [==============================] - 395s 2s/step - loss: 1.2274 - accuracy: 0.5530 - auc: 0.8419 - val_loss: 1.0735 - val_accuracy: 0.5311 - val_auc: 0.8460
Epoch 3/10
193/193 [==============================] - 394s 2s/step - loss: 1.1744 - accuracy: 0.5839 - auc: 0.8595 - val_loss: 1.2581 - val_accuracy: 0.4661 - val_auc: 0.7889
Epoch 4/10
193/193 [==============================] - 387s 2s/step - loss: 1.1409 - accuracy: 0.5963 - auc: 0.8684 - val_loss: 1.1622 - val_accuracy: 0.4615 - val_auc: 0.8166
Epoch 5/10
193/193 [==============================] - 384s 2s/step - loss: 1.1088 - accuracy: 0.5966 - auc: 0.8725 - val_loss: 1.1109 - val_accuracy: 0.5559 - val_auc: 0.8451
Epoch 6/10
193/193 [==============================] - 378s 2s/step - loss: 1.0854 - accuracy: 0.6070 - auc: 0.8797 - val_loss: 1.0463 - 

E0000 00:00:1787533748.101712      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


193/193 [==============================] - 472s 2s/step - loss: 1.7296 - accuracy: 0.4732 - auc: 0.7544 - val_loss: 0.9639 - val_accuracy: 0.6465 - val_auc: 0.8814
Epoch 2/60
193/193 [==============================] - 430s 2s/step - loss: 1.4863 - accuracy: 0.4842 - auc: 0.7790 - val_loss: 0.9818 - val_accuracy: 0.6200 - val_auc: 0.8761
Epoch 3/60
193/193 [==============================] - 443s 2s/step - loss: 1.3843 - accuracy: 0.4848 - auc: 0.7877 - val_loss: 1.0036 - val_accuracy: 0.5907 - val_auc: 0.8695
Epoch 4/60
193/193 [==============================] - 466s 2s/step - loss: 1.3066 - accuracy: 0.5051 - auc: 0.8089 - val_loss: 0.9965 - val_accuracy: 0.5897 - val_auc: 0.8709
Epoch 5/60
193/193 [==============================] - 470s 2s/step - loss: 1.2598 - accuracy: 0.5218 - auc: 0.8212 - val_loss: 0.9792 - val_accuracy: 0.6007 - val_auc: 0.8753
Epoch 6/60
193/193 [==============================] - 460s 2s/step - loss: 1.1996 - accuracy: 0.5325 - auc: 0.8332 - val_loss: 0.9633 - 